# 01 - Build and freeze a RAG corpus

Notebook này chỉ làm **Ingestion**. Nó nhận file, folder Kaggle Dataset hoặc ZIP; sau khi hoàn tất sẽ
đóng băng kết quả thành `corpus_bundle.zip`. Notebook không load answer model hay mở chat.


## 1. Cài đặt và import source

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/NhoDung/RAG-basic.git"
REPO_DIR = Path("/kaggle/working/RAG-basic")

if not (Path.cwd() / "rag_kaggle").exists():
    if REPO_DIR.exists():
        subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
    else:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
    os.chdir(REPO_DIR)

PROJECT_DIR = Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"])
print("Project:", PROJECT_DIR)


## 2. Chọn model trực tiếp và khai báo input

In [ ]:
# Mỗi phần nhận trực tiếp Hugging Face model ID hoặc tên PaddleOCR.
OCR_MODEL = "PaddleOCR-VL-1.6"
VISION_MODEL = None  # Ví dụ: "Qwen/Qwen2.5-VL-3B-Instruct"
EMBEDDING_MODEL = "BAAI/bge-m3"
EMBEDDING_REVISION = None
EMBEDDING_QUERY_INSTRUCTION = None

# Có thể trộn folder, ZIP và file đơn. Folder được duyệt đệ quy.
INPUT_SOURCES = [
    "/kaggle/input/my-documents",
    # "/kaggle/input/my-documents/documents.zip",
    # "/kaggle/input/my-documents/report.pdf",
]

WORK_DIR = Path("/kaggle/working/rag_ingestion")
OUTPUT_BUNDLE = Path("/kaggle/working/corpus_bundle.zip")
INCLUDE_SOURCE_DOCUMENTS = False


## 3. Cài PaddleOCR-VL trong môi trường riêng

In [ ]:
INSTALL_OCR = OCR_MODEL is not None
PADDLE_VERSION = "3.2.1"
PADDLE_CUDA = "cu126"
OCR_ENV = Path("/tmp/paddle_env")
OCR_PYTHON = None

if INSTALL_OCR:
    try:
        python = OCR_ENV / "bin" / "python"
        if not python.exists():
            subprocess.check_call([sys.executable, "-m", "venv", str(OCR_ENV)])
        pip = [str(python), "-m", "pip", "install", "-q"]
        subprocess.check_call(pip + ["--upgrade", "pip"])
        subprocess.check_call(pip + [
            f"paddlepaddle-gpu=={PADDLE_VERSION}",
            "-i", f"https://www.paddlepaddle.org.cn/packages/stable/{PADDLE_CUDA}/",
            "--extra-index-url", "https://pypi.org/simple",
        ])
        subprocess.check_call(pip + ["paddleocr[doc-parser]"])
        OCR_PYTHON = str(python)
    except Exception as exc:
        print("PaddleOCR-VL setup failed; OCR will be disabled:", exc)

print("OCR_PYTHON =", OCR_PYTHON)


## 4. Tạo ingestion runtime và kiểm tra tài nguyên

In [ ]:
from rag_kaggle import (
    IngestionPipeline, PipelineConfig, configure_ingestion_devices,
    inspect_resources, suggest_model_upgrades,
)

config = PipelineConfig.from_yaml(PROJECT_DIR / "configs" / "baseline.yaml")
config.work_dir = WORK_DIR
config.parsing.ocr_model_name = OCR_MODEL or "PaddleOCR-VL-1.6"
config.parsing.ocr_python = OCR_PYTHON
config.parsing.enable_ocr = OCR_MODEL is not None and OCR_PYTHON is not None
config.vision.enabled = VISION_MODEL is not None
if VISION_MODEL:
    config.vision.model = VISION_MODEL
config.retrieval.dense_model = EMBEDDING_MODEL
config.retrieval.dense_fallback_model = EMBEDDING_MODEL  # Không âm thầm đổi sang model khác.
config.retrieval.dense_revision = EMBEDDING_REVISION
config.retrieval.dense_query_instruction = EMBEDDING_QUERY_INSTRUCTION
config.artifacts.include_source_documents = INCLUDE_SOURCE_DOCUMENTS

resources = inspect_resources()
print("Resources:", resources)
print("Allocation:", configure_ingestion_devices(config))
print("Suggestions:", suggest_model_upgrades("ingestion", resources, config))
pipeline = IngestionPipeline(config)


## 5. Ingest, freeze và export

In [ ]:
report = pipeline.ingest_sources(INPUT_SOURCES, reset=True, progress=print)
print("Ingestion stats:", report["stats"])
if not report["ok"]:
    raise RuntimeError(report["errors"])

bundle = pipeline.export_corpus_bundle(OUTPUT_BUNDLE)
print("Frozen corpus:", pipeline.validate_corpus())
print("Bundle:", bundle, f"({bundle.stat().st_size / 1024**2:.1f} MB)")

from IPython.display import FileLink, display
display(FileLink(str(bundle)))
